# 1장 1강: 인공신경망과 DNN — 실습문제


## 실습 목표

- 콘크리트 데이터에서 모델에 넣는 입력과 맞히려는 실제 정답을 구분합니다.
- 퍼셉트론, 피드포워드 신경망(FFNN), 다층 퍼셉트론(MLP), DNN의 관계를 설명합니다.
- 입력층·은닉층·출력층의 역할과 각 층의 뉴런 수를 읽습니다.
- 작은 숫자 계산으로 가중치와 편향의 역할을 확인합니다.

## 실습 환경 / 데이터

- Python, pandas, NumPy를 사용합니다. pandas는 CSV 표를 읽고, NumPy는 숫자 묶음을 계산하는 데 사용합니다.
- 이 노트북과 **`concrete_data.csv`를 같은 폴더에 놓고**, 해당 폴더를 VS Code에서 연 뒤 노트북을 실행하세요.
- CSV는 제공받은 원본 파일을 사용합니다. 데이터 다운로드는 필요하지 않습니다.
- 파일명은 대소문자와 밑줄까지 같아야 합니다. `FileNotFoundError`가 나면 파일 위치와 이름부터 확인하세요.
- pandas와 NumPy가 설치된 Python 환경을 노트북의 커널로 선택하세요. 커널은 코드를 실제로 실행하는 Python 환경입니다.

## 이번 강의의 범위

제공된 1장 1강 목차인 **퍼셉트론·FFNN·MLP, 입력층·은닉층·출력층, 가중치·편향**을 연습합니다. 강의 본문은 제공되지 않았으므로, 아래의 신경망 크기와 가중치·편향 숫자는 이 범위를 설명하기 위해 정한 **실습용 예시**입니다.

이 실습에서는 **신경망의 구조와 뉴런 내부의 가중합 계산**까지 다룹니다. 활성화 함수는 뉴런의 출력 계산에 쓰인다는 역할만 읽습니다. Sigmoid·Tanh·ReLU의 공식과 다층 순전파 구현, 역전파, 학습·성능평가 코드는 다루지 않습니다. 계산된 `z`를 학습된 모델의 압축강도 예측값으로 해석하지 않습니다.

## 실습 구성

| 구분 | 세부 문제 | 할 일 | 연결되는 강의 내용 |
|---|---|---|---|
| 실습 1 | 1-1 | 데이터에서 입력·정답을 나누고 입력층·출력층 연결하기 | 입력층, 출력층 |
| 실습 2 | 2-1 | 한 뉴런의 가중합과 편향 계산하기 | 퍼셉트론의 기본 계산, 가중치·편향 |
| 실습 2 | 2-2 | 신경망 구조와 연결의 수 읽기 | FFNN·MLP·DNN, 은닉층, 파라미터 |
| 기초 과제 | 3-1 | 다른 행에 같은 계산을 적용하고 작은 구조 읽기 | 실습 1·2 복습 |

**실습 2개와 과제 1개**이며, 실습 2는 두 세부 문제로 나눴습니다. 과제에서는 새로운 방법을 찾아내지 않고, 실습에서 사용한 방법을 그대로 적용합니다.

## 진행 안내

1. 아래 ‘실습 준비’ 코드부터 위에서 아래로 실행합니다. 셀은 노트북의 글 또는 코드를 담는 한 칸입니다.
2. 문제의 표와 요구사항을 읽고, 코드 작성 셀에 한 단계씩 작성합니다.
3. 실행 결과뿐 아니라 수식·계산 과정과 해석 질문의 답변도 적습니다.
4. 배열에서 첫 번째 위치는 `0`, 세 번째 위치는 `2`로 표시합니다. 사람의 순서와 코드의 위치 번호를 구분하세요.
5. 이 실습의 가중치는 직접 정한 예시입니다. 데이터를 이용해 좋은 값을 찾는 ‘학습’은 아직 하지 않습니다.

## 시작 전에 읽을 용어

| 용어 | 쉽게 풀어쓴 뜻 | 이번 실습에서 연결하기 |
|---|---|---|
| 샘플 | 관측한 대상 한 개 | CSV의 한 행 |
| 특성 | 샘플을 설명하는 항목 | 시멘트량, 물의 양, 재령 등 |
| 입력 X | 계산에 사용하는 정보 | 정답 열을 제외한 8개 열 |
| 타깃 y | 모델이 맞히려는 실제 정답 | 측정된 압축강도 |
| 뉴런 | 입력에 가중치를 곱해 더하고 편향을 더한 뒤 출력 계산을 하는 단위 | 작은 계산 단위 하나 |
| 가중치 w | 각 입력에 곱하는 값 | 시멘트량에 곱하는 0.1 |
| 편향 b | 가중치를 곱해 더한 값에 추가하는 값 | 계산 마지막에 더하는 5 |
| 파라미터 | 모델 계산에 쓰며 학습으로 조정하는 값 | 가중치와 편향 |
| 입력층 | 입력 특성을 전달하는 시작 부분 | 전체 특성을 쓰면 입력 8개 |
| 은닉층 | 입력층과 출력층 사이에서 중간 계산을 하는 층 | 구조 예시의 뉴런 4개 층과 2개 층 |
| 출력층 | 최종적으로 필요한 출력을 만드는 층 | 압축강도 한 값을 내는 뉴런 1개 |
| Shape | 배열의 축별 크기 | (1030, 8)은 1030행·8열 |

용어를 외우기보다 해당 문제가 나올 때 다시 확인하세요.


### 뉴런 하나, 층 하나, 신경망 하나

**뉴런은 계산 단위 하나**, **층은 그런 계산 단위들을 모은 한 단계**, **신경망은 여러 단위를 연결한 전체 구조**입니다. 은닉층에 뉴런이 4개 있다는 말은 계산 단위 4개가 같은 단계에 있다는 뜻입니다. 은닉층이 4개 있다는 뜻과 다릅니다.

콘크리트 예측 상황에서는 재료 구성과 재령을 입력하고, 모델이 압축강도 한 값을 출력하도록 설계합니다. 데이터에 이미 기록된 압축강도는 나중에 모델의 출력과 비교할 **실제 정답**입니다. 출력층이 데이터의 정답을 그대로 꺼내 보여 주는 것은 아닙니다.

### 코드와 수식을 처음 읽을 때

| 표시 | 읽는 방법 | 예시 |
|---|---|---|
| 코드의 `=` | 오른쪽 결과를 왼쪽 이름에 저장 | `b = 5`는 b라는 이름으로 5를 사용한다는 뜻 |
| 수식의 `=` | 양쪽 값이 같음 | 2 + 3 = 5 |
| 코드의 `*` | 곱하기 | `2 * 3`은 2 × 3 |
| 수식의 x₁, x₂ | 첫 번째 입력, 두 번째 입력 | 아래 작은 숫자는 순서를 구분하는 번호이며 제곱이 아님 |
| 코드의 `[0]`, `[1]` | 첫 번째, 두 번째 위치의 값 선택 | Python의 위치 번호는 0부터 시작 |
| 변수 | 값에 붙인 이름 | `z`는 계산 결과를 다시 사용하기 위한 이름 |

**계산**은 지금 주어진 가중치·편향으로 결과를 구하는 일입니다. **학습**은 데이터를 이용해 가중치·편향을 조정하는 일입니다. 코드를 실행했다고 해서 반드시 학습한 것은 아닙니다. 이번 강의에서는 계산과 구조 읽기를 연습합니다.


## 실습 준비 — 제공 코드를 그대로 실행하세요

아래 코드는 CSV를 읽기 위한 준비 코드입니다. 학생이 새로 작성하거나 암기할 부분은 아닙니다.

원본에는 `fine_aggregate ` 열 이름 끝에 공백이 있습니다. `str.strip()`은 열 이름의 양 끝 공백만 제거합니다. 데이터의 숫자나 행 순서를 바꾸지 않습니다. 표의 앞부분을 확인한 뒤 문제를 시작하세요.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("concrete_data.csv")
df.columns = df.columns.str.strip()

print("전체 데이터 크기:", df.shape)
print("열 이름:", df.columns.tolist())
print(df.head(3).to_string(index=False))


전체 데이터 크기: (1030, 9)
열 이름: ['cement', 'blast_furnace_slag', 'fly_ash', 'water', 'superplasticizer', 'coarse_aggregate', 'fine_aggregate', 'age', 'concrete_compressive_strength']
 cement  blast_furnace_slag  fly_ash  water  superplasticizer  coarse_aggregate  fine_aggregate  age  concrete_compressive_strength
  540.0                 0.0      0.0  162.0               2.5            1040.0           676.0   28                          79.99
  540.0                 0.0      0.0  162.0               2.5            1055.0           676.0   28                          61.89
  332.5               142.5      0.0  228.0               0.0             932.0           594.0  270                          40.27


---

## 실습 1. 콘크리트 데이터와 신경망의 입력·출력 연결하기


### 문제 1-1. 입력 특성 8개와 실제 정답 1개 구분하기

#### 문제 설명

콘크리트의 재료 구성과 재령을 보고 압축강도를 추정하려는 상황입니다. 이 문제에서는 모델을 학습하지 않고 **어떤 정보를 넣고 어떤 값을 출력해야 하는지** 정리합니다.

| CSV 열 이름 | 의미 | 역할 |
|---|---|---|
| cement | 시멘트량 | 입력 |
| blast_furnace_slag | 고로 슬래그량 | 입력 |
| fly_ash | 플라이애시량 | 입력 |
| water | 물의 양 | 입력 |
| superplasticizer | 고성능 감수제량 | 입력 |
| coarse_aggregate | 굵은 골재량 | 입력 |
| fine_aggregate | 잔골재량 | 입력 |
| age | 재령: 경과한 양생 일수 | 입력 |
| concrete_compressive_strength | 측정된 콘크리트 압축강도 | 실제 정답 |

#### 먼저 이해하기

- CSV의 한 행은 한 샘플입니다. 입력 특성은 한 샘플을 설명하는 8개 값입니다.
- 전체 입력 배열을 `X`, 실제 압축강도를 모은 배열을 `y`라고 부릅니다.
- 이 실습에서는 입력 특성 하나를 입력층의 한 자리와 연결합니다. 압축강도라는 연속적인 수치 하나를 내므로 출력층은 뉴런 1개로 설정합니다.
- 실제 정답 `y`를 모델의 입력 `X`에 함께 넣지 않습니다. 예측하려는 정답 자체를 미리 알려주는 것이 되기 때문입니다.

#### 읽기 도움말 — 입력과 정답을 짝지어 보기

한 행은 한 콘크리트 샘플에 대한 기록입니다. 같은 행에서 재료·재령을 고르면 입력이 되고, 압축강도를 고르면 실제 정답이 됩니다. 열을 나누더라도 행의 순서는 유지해야 같은 샘플끼리 짝이 맞습니다. 이번 코드에서는 행을 섞거나 삭제하지 않습니다.

`X.shape`는 배열의 크기를 보여 주는 정보입니다. X 안에 들어 있는 모든 숫자의 합을 구하는 명령이 아닙니다. 먼저 ‘샘플이 몇 개 있는가’, 다음으로 ‘샘플 하나에 입력이 몇 개 있는가’를 읽으세요.

#### 요구사항

1. 아래 제공된 열 목록을 사용해 입력 배열 `X`와 타깃 배열 `y`를 만드세요.
2. `X.shape`, `y.shape`를 출력하세요.
3. 첫 번째 샘플의 입력 `X[0]`과 실제 정답 `y[0]`을 출력하세요.
4. 전체 입력 8개를 사용하는 신경망의 입력층과 출력층 뉴런 수를 답변란에 적으세요.

#### 작성 도움말

제공된 `feature_columns`의 순서를 그대로 사용하세요. `df[feature_columns]`는 해당 열들을 고르고, `.to_numpy()`는 그 숫자를 NumPy 배열로 바꿉니다. `X[0]`은 첫 행, `y[0]`은 첫 번째 정답입니다.

#### 해석 질문

**Q1.** `X.shape`와 `y.shape`는 각각 무엇이며, 각 숫자는 무엇을 뜻하나요?

**Q2.** 첫 샘플의 시멘트량, 물의 양, 재령, 실제 압축강도는 각각 얼마인가요?

**Q3.** 입력층은 왜 8개이고 출력층은 왜 1개인가요? 샘플이 1030개이면 입력층도 1030개인가요?

**Q4.** 실제 압축강도 열을 `X`에 넣지 않는 이유는 무엇인가요? 지금 코드에서 압축강도를 예측했나요?

#### 제출 결과

- X와 y의 Shape, 첫 샘플의 입력과 정답 출력
- 입력층·출력층 뉴런 수 및 Q1~Q4 답변


In [2]:
# 열 순서는 바꾸지 마세요. 이 목록은 제공 코드입니다.
feature_columns = [
    "cement", "blast_furnace_slag", "fly_ash", "water",
    "superplasticizer", "coarse_aggregate", "fine_aggregate", "age"
]
target_column = "concrete_compressive_strength"

# 아래부터 작성하세요.
# 1. X = df[feature_columns].to_numpy() 형식을 사용하세요.
# 2. y = df[target_column].to_numpy() 형식을 사용하세요.
# 3. Shape와 첫 번째 샘플을 출력하세요.
X = df[feature_columns].to_numpy()
y = df[target_column].to_numpy()

print('X shape', X.shape)
print('Y shape', y.shape)
print('첫 번째 입력',X[0])
print('첫 번째 실제 정답', y[0])

X shape (1030, 8)
Y shape (1030,)
첫 번째 입력 [ 540.     0.     0.   162.     2.5 1040.   676.    28. ]
첫 번째 실제 정답 79.99


#### 답변 작성란

**수식 및 계산 과정 / 구조 설명**

- 작성: 

**Q1. `X.shape`와 `y.shape`는 각각 무엇이며, 각 숫자는 무엇을 뜻하나요?**

- 답변: X.shape는 (1030,8) 한 행이 한 샘플, 1030개는 샘플 수(고객 수), 각 행에 8개의 입력값(특징)을 가지고 있음
- y.shape는 (1030,) y는 각 샘플에 대한 정답 1개씩 1030개가 한 줄의 숫자 배열로 담겨있는 행렬

**Q2. 첫 샘플의 시멘트량, 물의 양, 재령, 실제 압축강도는 각각 얼마인가요?**

- 답변: 시멘트량 540, 물의 양 162, 재령은 28, 실제 압축강도는 79.99

**Q3. 입력층은 왜 8개이고 출력층은 왜 1개인가요? 샘플이 1030개이면 입력층도 1030개인가요?**

- 답변: 한 샘플에서 사용할 특성이 8개라 입력층의 자리를 8개
- 최종적으로 내보낼 수치가 압축강도 하날라 출력층의 뉴런을 1개로 정한다.
- 샘플 1030개는 이 같은 구조에 넣을 사례의 개수, 
- 학생 수가 달라지더라도 학생 한 명에게 긱록하는 항목수는 그대로 일 수 있는 것처럼
- 샘플 수와 특성 수를 따로 세는 것이 일반적

**Q4. 실제 압축강도 열을 `X`에 넣지 않는 이유는 무엇인가요? 지금 코드에서 압축강도를 예측했나요?**

- 답변: 압축강도(y)는 재료, 재령으로부터 알아내려고 하는 결과, 
- 예측하려는 시점에서는 그 정답을 모른다고 가정하는 것이 더 정확하기 때문에 정답 자체를 x(입력층)으로 넣지 않는다.


---

## 실습 2. 뉴런의 계산과 여러 층의 구조 이해하기


### 문제 2-1. 가중치와 편향을 사용한 한 뉴런의 계산

#### 문제 설명

전체 8개 특성을 한꺼번에 계산하면 길어지므로 **첫 번째 샘플의 시멘트량과 물의 양 두 개만** 사용합니다. 아래 가중치와 편향은 계산 연습을 위해 정한 값이며, 학습으로 얻은 값이 아닙니다.

| 기호 | 뜻 | 값 |
|---|---|---:|
| x₁ | 첫 샘플의 시멘트량 | 540 |
| x₂ | 첫 샘플의 물의 양 | 162 |
| w₁ | 시멘트량에 곱할 가중치 | 0.1 |
| w₂ | 물의 양에 곱할 가중치 | -0.2 |
| b | 마지막에 더할 편향 | 5 |

#### 먼저 이해하기 — 퍼셉트론과 뉴런

퍼셉트론은 신경망의 기본 계산 구조를 이해하는 출발점입니다. 고전적인 퍼셉트론은 입력에 가중치를 곱해 더하고 편향을 더한 뒤, 기준에 따라 0 또는 1을 출력합니다.

이 문제에서는 **그 내부의 가중합과 편향 계산만** 연습합니다. 콘크리트 압축강도를 0·1로 바꾸거나 퍼셉트론 분류를 구현하지 않습니다. 일반적인 신경망 뉴런은 이 계산에 활성화 함수를 적용해 출력을 만들 수 있으며, 활성화 함수의 구체적인 공식은 다음 강의 범위입니다.

$$z = w_1x_1 + w_2x_2 + b$$

#### 수식을 말로 읽기

- `w₁x₁`: 첫 입력값에 첫 가중치를 곱합니다.
- `w₂x₂`: 둘째 입력값에 둘째 가중치를 곱합니다.
- 두 곱셈 결과를 더하고, 마지막에 편향 b를 더합니다.
- `z`는 활성화 함수를 적용하기 전의 계산값입니다. 실제 정답 y와 구분합니다.

#### 읽기 도움말 — 퍼셉트론의 전체 흐름

퍼셉트론의 기본 흐름은 **입력 받기 → 입력마다 가중치 곱하기 → 모두 더하고 편향 더하기 → 기준에 따라 출력 결정하기**입니다. 이 문제는 마지막 출력 결정 직전까지 계산합니다.

기준을 0으로 정하고 ‘계산값이 0 이상이면 1, 0 미만이면 0’이라고 약속한 퍼셉트론을 생각해 보세요. 계산값이 2라면 출력은 1이고, -2라면 출력은 0입니다. 이때 0과 1은 두 종류를 구분하기 위한 표시입니다. 콘크리트 압축강도의 실제 수치와는 다른 종류의 출력입니다. 이 설명은 퍼셉트론의 역할을 이해하기 위한 것이며, 분류 코드를 추가로 작성할 필요는 없습니다.

수식의 `w₁x₁`은 두 기호를 이어 쓴 곱셈입니다. Python에서는 곱셈 기호를 생략할 수 없으므로 `w[0] * x_small[0]`처럼 씁니다. 수식의 첫 번째 번호 1과 코드의 첫 위치 번호 0이 같은 항을 가리킨다는 점을 확인하세요.

#### 요구사항

1. 제공된 방법으로 첫 샘플의 시멘트량과 물의 양을 `x_small`에 담고 출력하세요.
2. `w = np.array([0.1, -0.2])`, `b = 5.0`을 준비하세요.
3. 두 곱셈 결과를 `part1`, `part2`에 저장하고 출력하세요.
4. `z = part1 + part2 + b`를 계산하고 출력하세요.
5. 입력과 가중치를 유지하고 편향만 0으로 바꾼 계산값 `z_b0`를 출력하세요.
6. 두 가중치를 모두 0으로 두고 편향 5를 유지한 계산값 `z_w0`를 출력하세요.

#### 작성 도움말

`df[["cement", "water"]].to_numpy()[0]`은 두 열을 고른 뒤 첫 번째 행을 가져옵니다. `x_small[0]`은 시멘트량, `x_small[1]`은 물의 양입니다. `part1 = w[0] * x_small[0]`처럼 같은 위치의 값끼리 곱하세요.

#### 해석 질문

**Q1.** 두 곱셈 결과와 최종 z는 각각 얼마인가요? 대입부터 덧셈까지 적으세요.

**Q2.** 편향을 5에서 0으로 바꾸면 무엇이 얼마나 달라지나요?

**Q3.** 모든 가중치가 0이면 z는 무엇으로 결정되나요? 편향도 0일 수 있나요?

**Q4.** z를 첫 샘플의 실제 정답 79.99 또는 학습된 예측값이라고 부를 수 있나요?

**Q5.** 이 코드가 고전적인 퍼셉트론의 전체 동작을 구현한 것인가요?

#### 제출 결과

- 입력, 두 곱셈 결과, z, z_b0, z_w0 출력
- 수식 대입·계산 과정 및 Q1~Q5 답변


In [3]:
# 첫 샘플의 두 특성을 가져오는 제공 코드입니다.
x_small = df[["cement", "water"]].to_numpy()[0]

# 아래부터 작성하세요.
# 1. x_small을 출력하고 w, b를 준비하세요.
# 2. 두 입력에 대응하는 가중치를 각각 곱하세요.
# 3. z, 편향이 0인 경우, 가중치가 모두 0인 경우를 계산하세요.
w = np.array([0.1, -0.2])
b = 5.0
part1 = w[0] * x_small[0]
part2 = w[1] * x_small[1]
z= part1 + part2 + b

# 편향값만 0으로 수정한다.
z_b0 = part1 + part2 + 0.0

# 가중치를 0으로 수정한다.
z_w0 = 0.0 * x_small[0] + 0.0 * x_small[1] + b

print('입력 [시멘트량, 물의 양]:', x_small)
print('첫 곱셈 결과: ', round(part1, 3))
print('둘째 곱셈 결과: ', round(part2, 3))
print('편향을 더한 z: ', round(z,3))
print('편향이 0인 경우: ', round(z_b0, 3))
print('가중치가 모두 0인 경우: ', round(z_w0, 3))

입력 [시멘트량, 물의 양]: [540. 162.]
첫 곱셈 결과:  54.0
둘째 곱셈 결과:  -32.4
편향을 더한 z:  26.6
편향이 0인 경우:  21.6
가중치가 모두 0인 경우:  5.0


#### 답변 작성란

**수식 및 계산 과정 / 구조 설명**

- 작성: 

**Q1. 두 곱셈 결과와 최종 z는 각각 얼마인가요? 대입부터 덧셈까지 적으세요.**

- 주어진 식은 z = w1x1 + w2x2 + b입니다.
- 값을 대입하면 z = 0.1 x 540 + (-0.2) x 162 + 5
- 1. 먼저 곱셈 = 54
- 2. 둘째 항 계산 = -32.4
- 3. 두 항을 더하기 = 54 + (-32.4) = 21.6
- 4. 편향 5 더하기 = z = 26.6

**Q2. 편향을 5에서 0으로 바꾸면 무엇이 얼마나 달라지나요?**

- 첫 항 54와 둘째 항 -32.4는 그대로 두고 마지막 덧셈만 +5에서 0으로 달라짐
- z = 26.6에서 21.6으로 5감소 

**Q3. 모든 가중치가 0이면 z는 무엇으로 결정되나요? 편향도 0일 수 있나요?**

- 답변: 가중치가 모두 0이면 0 x 540 + 0 x 162 + b = b가 되어 z = b가 된다.
- 즉, b가 0이면 z도 0
- 모든 가중치가 0인 상태에서 입력값이 달라져도 뉴런의 z는 b랑 같다.

**Q4. z를 첫 샘플의 실제 정답 79.99 또는 학습된 예측값이라고 부를 수 있나요?**

- 답변: No, 79.99는 측정해서 데이터에 기록한 실제 정답, y, 26.6은 정해둔 가중치, 편향으로 얻은 z
- z는 이 문제에서 활성화 함수를 적용하기 전에 값

**Q5. 이 코드가 고전적인 퍼셉트론의 전체 동작을 구현한 것인가요?**

- 답변: 아닙니다. 입력에 가중치를 곱하고 모두 더한 뒤 편향을 더하는 단계까지만 구현
- 고전적인 퍼셉트론은 그 다음에 기준과 비교하여 0 또는 1을 정하는 단계가 추가적으로 있음
- 예를 들어 기준이 0이고 0 이상일 때 1을 내도록 정했다면 양수인 z는 출력 1로 바뀐다
- z라는 계산값과 그 뒤에 결정한 출력 부분은 구분할 수 있어야 한다.


### 문제 2-2. 입력층·은닉층·출력층과 FFNN·MLP·DNN 구분하기

#### 문제 설명

전체 8개 입력 특성을 사용하는 다음 신경망 구조를 읽어 보세요. 구조의 크기는 실습용 예시이며 데이터에서 자동으로 정해진 정답이 아닙니다.

**입력층 8개 → 은닉층 1의 뉴런 4개 → 은닉층 2의 뉴런 2개 → 출력층 뉴런 1개**

각 층의 뉴런은 바로 앞 층의 모든 값과 연결된다고 가정합니다. 연결 하나마다 가중치 하나가 있으며, 은닉층과 출력층의 뉴런마다 편향 하나가 있습니다. 입력층에는 이 실습에서 학습할 가중치·편향을 따로 두지 않습니다.

#### 먼저 이해하기

| 이름 | 구분하는 기준 | 이 구조에서 확인하기 |
|---|---|---|
| FFNN: 피드포워드 신경망 | 정보가 입력에서 출력 쪽으로 흐르고 이전 단계로 되돌아가는 순환 연결이 없음 | 8 → 4 → 2 → 1의 방향 |
| MLP: 다층 퍼셉트론 | 은닉층을 포함하며, 보통 층 사이가 완전연결된 신경망 | 앞 층의 모든 값과 연결되는 여러 뉴런 |
| DNN: 심층 신경망 | 여러 은닉층을 쌓은 깊은 신경망 | 은닉층이 2개인 예시 |

이 이름들은 서로 완전히 별개의 선택지가 아닙니다. **여러 은닉층을 가진 MLP는 FFNN이면서 DNN일 수 있습니다.** FFNN 전체가 모두 완전연결 MLP인 것은 아니며, 깊다고 부르는 층 수의 기준은 문맥에 따라 다를 수 있습니다. 여기서는 여러 은닉층을 쌓는다는 뜻을 이해하면 됩니다.

실제 MLP에서는 활성화 함수를 통한 계산도 필요합니다. 이 문제의 화살표는 구조를 표시한 것이며, 단순 가중합을 여러 번 계산하는 것만으로 일반적인 MLP 전체를 구현했다고 보지 않습니다.

#### 수식을 말로 읽기

연결 수를 세기 위한 간단한 곱셈입니다.

- 한 층의 가중치 수 = **들어오는 값의 수 × 그 층의 뉴런 수**
- 한 층의 편향 수 = **그 층의 뉴런 수**
- 파라미터 수 = **가중치 수 + 편향 수**

예를 들어 8개 입력을 받는 뉴런 하나에는 가중치 8개가 필요합니다. 그런 뉴런이 4개라면 가중치는 8×4개입니다. 숫자를 외우지 말고 연결을 센다고 생각하세요.

#### 읽기 도움말 — 은닉층에서 무엇이 달라지나요?

입력층은 측정된 정보를 받고, 은닉층은 그 정보를 조합해 다음 층에 전달할 중간 값을 만듭니다. ‘은닉’은 데이터에 정답으로 직접 적혀 있지 않은 **중간 계산 단계**라는 뜻으로 이해하세요. 계산 결과를 확인할 수 없다는 뜻은 아닙니다.

첫 은닉층의 뉴런들이 같은 입력 8개를 받더라도, 각 뉴런은 자기 가중치와 편향을 사용합니다. 따라서 서로 다른 중간 값을 만들 수 있습니다. 다음 은닉층은 이 중간 값들을 입력으로 받습니다. 이때 전달되는 것은 계산 결과이며, 다음 층에 쓸 새로운 가중치를 계산해 전달하는 것이 아닙니다.

| 구분 | 확인할 내용 |
|---|---|
| 뉴런 수 | 한 층 안에 계산 단위가 몇 개 있는지 |
| 은닉층 수 | 입력층과 출력층 사이에 중간 계산 단계가 몇 개 있는지 |
| 완전연결 | 바로 앞 층의 모든 값이 그다음 층의 각 뉴런에 연결되는 방식 |

MLP는 Multi-Layer Perceptron, FFNN은 Feedforward Neural Network, DNN은 Deep Neural Network의 약자입니다. 현대적인 MLP의 뉴런은 고전적인 퍼셉트론처럼 언제나 0 또는 1만 내는 것이 아닙니다. 뉴런의 출력 계산에 쓰이는 활성화 함수는 다음 강의에서 구체적으로 배웁니다.

#### 요구사항

1. 답변란에 각 층의 이름·뉴런 수·역할을 적으세요.
2. 코드 셀에서 각 층의 가중치 수와 편향 수를 곱셈·덧셈으로 계산해 출력하세요.
3. 전체 가중치 수, 전체 편향 수, 전체 파라미터 수를 출력하세요.
4. 아래 Q1~Q4에 답하세요. 실제 신경망 객체를 만들거나 학습할 필요는 없습니다.

#### 작성 도움말

첫 은닉층은 `h1_weights = 8 * 4`, `h1_biases = 4`로 셉니다. 둘째 은닉층으로 들어오는 값은 원본 특성 8개가 아니라 바로 앞 층에서 나온 값 4개입니다. 출력층에는 그 앞 층의 값 2개가 들어옵니다.

#### 해석 질문

**Q1.** 각 층의 뉴런 수와 역할은 무엇인가요? 은닉층 4개·2개는 반드시 이 숫자여야 하나요?

**Q2.** 이 구조를 FFNN, MLP, DNN이라고 설명할 수 있는 이유를 각각 적으세요.

**Q3.** 층별 가중치·편향 수와 전체 파라미터 수는 얼마인가요? 샘플 수 1030을 곱해야 하나요?

**Q4.** 입력층을 제외하고 가중치·편향이 있는 계산층은 몇 개이며, 압축강도 한 값을 내는 출력층도 파라미터를 가지나요?

#### 제출 결과

- 각 층의 이름·뉴런 수·역할
- 층별 가중치·편향 수와 전체 합 출력
- 계산 과정 및 Q1~Q4 답변


In [4]:
# 구조: 입력 8 → 은닉 4 → 은닉 2 → 출력 1
# 첫 은닉층은 h1_weights = 8 * 4, h1_biases = 4로 계산합니다.
# 나머지 층과 전체 합을 같은 방식으로 계산하고 출력하세요.

# 입력층의 8은 원본 특성의 수
# 시멘트량 ~ 재령까지 8개의 특성 수
h1_weights = 8 * 4
h1_biases = 4

# 첫 은닉층의 가중치와 편향을 센다.
# 뉴런 하나는 입력 8개를 받는데 각각 곱할 가중치 8개와 마지막 더할 편향 1개가 필요하다
# 뉴런이 4개니까 가중치는  8 * 4, 편향은 4개다.

# 두 번째 은닉층은 앞 층의 출력 4개를 받는다.
# 각 뉴런은 원래 특성인 8개의 값을 받는 것이 아니라 첫 은닉층에서 전달된 4개의 값을 받는다.
h2_weights = 4 * 2
h2_biases = 2

# 출력층의 뉴런 하나도 계산을 한다.
# 두번 째 은닉층의 값 2개를 받기 때문에 가중치 2개, 편향 1개가 필요하다.
out_weights = 2 * 1
out_biases = 1

total_weights = h1_weights + h2_weights + out_weights
total_bias = h1_biases + h2_biases + out_weights
total_parameter = total_weights + total_bias

print('은닉층 1 가중치 / 편향 : ', h1_weights, h1_biases)
print('은닉층 2 가중치 / 편향 : ', h2_weights, h2_biases)
print('출력층 가중치 / 편향 :', out_weights, out_biases)
print('전체 가중치 수: ', total_weights)
print('전체 편향 수: ', total_bias)
print('전체 파라미터 수: ', total_parameter)


은닉층 1 가중치 / 편향 :  32 4
은닉층 2 가중치 / 편향 :  8 2
출력층 가중치 / 편향 : 2 1
전체 가중치 수:  42
전체 편향 수:  8
전체 파라미터 수:  50


#### 답변 작성란

**수식 및 계산 과정 / 구조 설명**

- 작성: 

**Q1. 각 층의 뉴런 수와 역할은 무엇인가요? 은닉층 4개·2개는 반드시 이 숫자여야 하나요?**

- 답변: 입력층의 8개의 자리에는 원본 특성 8개가 들어감
- 첫 은닉층의 뉴런 4개는 입력을 조합해 중간 값 4개를 만들고
- 두 번째 은닉층의 뉴런 2개는 그 4개를 받아 중간 값 2개를 만든다.
- 출력층의 뉴런 1개는 그 2개의 값을 받아 최종 압축강도 1개의 값을 만들도록 설계한 부분

**Q2. 이 구조를 FFNN, MLP, DNN이라고 설명할 수 있는 이유를 각각 적으세요.**

- 답변: FFNN인 이유: 출력 계산에 쓰이는 정보가 입력층에서 출력층으로 향하며 이전 단계로 돌아오는 순환 연결이 없기 때문
- MLP인 이유: 은닉층을 포함하고 인접 층 사이가 완전 연결된 다층 구조
- DNN인 이유: 여러 은닉층을 쌓아 중간 계산 단계를 늘렸기 때문

**Q3. 층별 가중치·편향 수와 전체 파라미터 수는 얼마인가요? 샘플 수 1030을 곱해야 하나요?**

- 답변: 첫 은닉층은 입력 8개를 받는 뉴런이 4개라 가중치 8 X 4 = 32개, 편향은 4개
- 두 번째 은닉층은 앞 층의 값 4개를 받는 뉴런이 2개라 가중치 4 X 2 = 8개, 편향은 2개
- 출력층은 앞 층의 값 2개를 받는 뉴런이 1개라 가중치 2개, 편향 1개
- 전체 가중치는 32 + 8 + 2 = 42
- 전체 편향은 4 + 2 + 1 = 7개
- -> 총 49개의 파라미터를 가진다.
- 같은 모델의 파라미터를 여러 샘플에 공통적으로 쓰기 때문에 샘플 수에 1030을 곱하지 않는다.

**Q4. 입력층을 제외하고 가중치·편향이 있는 계산층은 몇 개이며, 압축강도 한 값을 내는 출력층도 파라미터를 가지나요?**

- 답변: 첫 은닉층, 두 번째 은닉층, 출력층으로 계산층은 3개
- 입력층은 데이터를 받는 부분으로 별도 학습 파라미터를 세지 않는다.
- 출력층도 앞 층에서 받은 값들을 조합하여 계산을 하기 때문에 가중치 2개와 편향 1개, 총 3개의 파라미터가 필요하다.
- 출력 1개는 계산 결과, 파라미터 3개는 그 계산에 사용하는 값의 수를 의미한다.


---

## 과제. 실습에서 배운 계산과 구조 읽기 복습


### 문제 3-1. 세 번째 샘플의 계산과 작은 신경망 구조 확인

#### 문제 설명

실습 2-1과 같은 방법을 **세 번째 샘플**에 적용합니다. 입력은 시멘트량·물의 양 두 개이고, 가중치 `[0.1, -0.2]`와 편향 5는 실습과 같습니다.

| 항목 | 세 번째 샘플의 값 |
|---|---:|
| 시멘트량 | 332.5 |
| 물의 양 | 228 |
| 실제 압축강도 | 40.27 |

이어서 이 두 특성을 사용하는 구조 **입력 2 → 은닉 2 → 은닉 2 → 출력 1**을 읽습니다. 각 층은 앞 층의 모든 값과 연결되고 각 계산 뉴런에는 편향 하나가 있습니다.

#### 수식을 말로 읽기

`z = w₁×x₁ + w₂×x₂ + b`에 이번 샘플의 숫자만 바꿔 넣습니다. 가중치와 편향은 동일하게 사용합니다. 파라미터 수는 실습 2-2처럼 연결의 수와 뉴런별 편향의 수를 세어 구합니다.

#### 읽기 도움말 — 두 활동을 구분하세요

먼저 한 뉴런의 계산에 다른 샘플을 넣어 봅니다. 이어서 여러 층을 가진 구조의 가중치·편향 **개수**를 셉니다. 앞에서 구한 z가 뒤에 제시된 전체 신경망의 최종 출력이라는 뜻은 아닙니다. 실습에서 했던 계산 순서와 개수 세기 규칙을 그대로 사용하면 됩니다.

#### 요구사항

1. 제공 코드를 사용해 세 번째 샘플의 두 입력과 실제 정답을 가져와 출력하세요.
2. 두 곱셈 결과를 각각 저장한 다음 z를 구하여 출력하세요.
3. 편향만 0으로 바꾼 z_b0를 출력하세요.
4. `2 → 2 → 2 → 1`의 층별 가중치·편향 수와 전체 파라미터 수를 계산해 출력하세요.
5. 계산 과정을 적고 Q1~Q5에 답하세요.

#### 작성 도움말

세 번째 샘플은 `[2]`로 가져옵니다. 실습 2-1의 계산 코드를 참고하되, 첫 번째 샘플의 540·162를 다시 사용하지 마세요. 첫 은닉층의 가중치 수는 `2 * 2`입니다. 다음 층도 같은 규칙으로 셉니다.

#### 해석 질문

**Q1.** 세 번째 입력과 실제 정답은 무엇인가요? 왜 위치 번호 `[2]`를 사용하나요?

**Q2.** 두 곱셈 결과, z, z_b0는 각각 얼마인가요? 계산 과정을 적으세요.

**Q3.** z가 실제 정답과 다르거나 음수이면 코드가 반드시 잘못된 것인가요?

**Q4.** `2 → 2 → 2 → 1`의 입력층·은닉층·출력층과 전체 파라미터 수를 설명하세요.

**Q5.** 샘플이 달라질 때 이번 코드에서 바꾼 것과 유지한 것은 무엇인가요? 학습을 수행했나요?

#### 제출 결과

- 세 번째 샘플의 두 입력과 실제 정답 출력
- 두 곱셈 결과, z, z_b0 출력과 손계산
- 층별 가중치·편향 수와 전체 파라미터 수
- Q1~Q5 답변


In [ ]:
# 세 번째 샘플을 가져오는 제공 코드입니다.
task_x = df[["cement", "water"]].to_numpy()[2]
task_y = df["concrete_compressive_strength"].to_numpy()[2]

# 아래부터 작성하세요.
# 실습 2-1: 같은 가중치와 편향으로 task_x를 계산합니다.
# 실습 2-2: 2 → 2 → 2 → 1의 가중치·편향 개수를 셉니다.



#### 답변 작성란

**수식 및 계산 과정 / 구조 설명**

- 작성: 

**Q1. 세 번째 입력과 실제 정답은 무엇인가요? 왜 위치 번호 `[2]`를 사용하나요?**

- 답변: 

**Q2. 두 곱셈 결과, z, z_b0는 각각 얼마인가요? 계산 과정을 적으세요.**

- 답변: 

**Q3. z가 실제 정답과 다르거나 음수이면 코드가 반드시 잘못된 것인가요?**

- 답변: 

**Q4. `2 → 2 → 2 → 1`의 입력층·은닉층·출력층과 전체 파라미터 수를 설명하세요.**

- 답변: 

**Q5. 샘플이 달라질 때 이번 코드에서 바꾼 것과 유지한 것은 무엇인가요? 학습을 수행했나요?**

- 답변:


---

## 실습 마무리

**1. 퍼셉트론 내부에서 가중치와 편향은 각각 어떻게 사용하나요?**

- 답변: 각 입력에 해당 가중치를 곱하고 모두 더한 뒤 편향을 더한다.
- 고전적인 퍼셉트론은 그 뒤 기준에 따라 0 또는 1의 출력을 정한다.

**2. 입력층·은닉층·출력층의 역할을 각각 한 문장으로 설명하세요.**

- 답변: 입력층은 입력 정보를 전달한다(데이터 특성 전달)
- 은닉층은 입력층과 출력층 사이에서 중간 계산을 한다.
- 출력층은 문제에서 필요한 최종 출력을 만든다.

**3. 하나의 신경망이 FFNN이면서 MLP이고 DNN일 수 있나요?**

- 답변:  가능
- 순환 연결이 없는 피드포워드 구조이면서 완전연결된 은닉층을 여러 개 쌓았다면 세 표현으로 같은 구조의 다른 특성을 설명 가능

**4. 같은 뉴런의 가중치와 편향을 샘플마다 새로 만들었나요?**

- 답변: 아닙니다. 실습과 과제의 두 특성 계산에는 같은 가중치와 편향을 사용함, 모델의 같은 파라미터를 여러 샘플에 적용함

**5. 이번 실습에서 신경망 학습이나 완전한 다층 순전파를 수행했나요?**

- 답변: 아닙니다. 입력과 정답을 구분하고 한 뉴런의 가중합, 편향을 계산하고 다층 구조를 읽었다.
- 전체 다층 순전파와 학습은 구현하지 않음
